# Corrected NIBSS Fraud Dataset Generator - Build & Validation

1.   List item
2.   List item



This notebook builds a leakage-corrected version of the NIBSS-calibrated synthetic
fraud dataset, then runs the same forensic validation tests used to identify the
original dataset's flaws, plus a difficulty sanity check.

**Structure:**
1. Generator (design decisions documented against each specific finding they fix)
2. Validation: leakage fix, merchant_risk_score coherence, null bank/geography, NIBSS-calibrated fraud amounts
3. Difficulty sanity check: confirms the corrected dataset is realistically hard, not trivially easy

Run all cells top to bottom. On Colab, this takes a few minutes end to end.

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from scipy import stats

SEED = 42
rng = np.random.default_rng(SEED)
print("Setup OK")

Setup OK


## 1. Generator - design constants

Each constant below is commented against the specific reverse-engineering finding it addresses.

In [ ]:


N_CUSTOMERS = 10000
DAYS = 365
START_DATE = datetime(2023, 1, 1)

CHANNELS = ['ATM', 'ECOM', 'IB', 'Mobile', 'POS', 'Web']
BANKS = [f'Bank_{i}' for i in range(1, 11)]
LOCATIONS = ['Lagos', 'Abuja', 'Ogun', 'Oyo', 'Rivers', 'Other']
AGE_GROUPS = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
MERCHANT_CATS = ['Medical','ATM Withdrawal','Retail','Grocery','Education',
                  'Transfer','Electronics','Transport','Airtime','Fashion',
                  'Restaurant','Bill Payment','Entertainment']
FRAUD_TECHNIQUES = ['SOCIAL_ENGINEERING','ROBBERY','CARD_THEFT','OTHER',
                     'PIN_COMPROMISE','PHISHING']

# ---------------------------------------------------------------------------
# FIX #6 & #10 (RE findings): channel affects fraud probability in the
# original data (chi2 p=4.1e-7) -- this is legitimate for NIBSS calibration,
# but must be TRANSPARENTLY DOCUMENTED rather than left implicit, so a paper
# using this data can correctly state "channel effects are calibrated into
# generation, not independently discovered." Values below are deliberately
# chosen to be directionally consistent with NIBSS's own reported channel
# fraud shares, not tuned to any specific downstream ML result.
CHANNEL_FRAUD_FACTOR = {
    'ATM': 0.25, 'ECOM': 0.55, 'IB': 0.70, 'Mobile': 1.30, 'POS': 1.25, 'Web': 1.20
}

# FIX #11: month effects were extremely strongly encoded in the original
# (chi2 p=5.4e-17). Kept here, again deliberately, for NIBSS seasonal
# calibration (e.g. higher fraud around Dec/Jan festive spending), but at a
# MUCH weaker magnitude than the original (which produced an implausible
# ~4x swing between December and May) -- and documented explicitly rather
# than left for a downstream paper to "rediscover" as a novel finding.
MONTH_FRAUD_FACTOR = {1:1.15, 2:1.05, 3:1.00, 4:1.00, 5:1.10, 6:0.95,
                       7:1.05, 8:0.95, 9:1.00, 10:0.95, 11:0.90, 12:1.10}

# FIX #12/13: bank and location showed NO statistically significant
# association with fraud in the reverse-engineered original (p=0.57-0.88).
# This generator deliberately assigns bank and location WITHOUT any fraud
# effect, so that a paper using this dataset cannot spuriously "discover"
# bank- or geography-based fraud ecosystems that were never real.
BASE_FRAUD_RATE = 0.0033  # tuned so realised rate lands near the real ~0.3% NIBSS-consistent target

# FIX #5 (merchant_risk_score): in the original, this lookup was UNCORRELATED
# (r=-0.197) with actual generated fraud rates -- internally incoherent as a
# "risk score." Here it is instead DERIVED FROM the same generation logic
# used for fraud injection, so it is a genuine, internally consistent risk
# signal rather than arbitrary noise dressed up as a score.
MERCHANT_FRAUD_FACTOR = {
    'Medical':0.3,'ATM Withdrawal':0.4,'Retail':0.7,'Grocery':0.5,'Education':0.4,
    'Transfer':1.4,'Electronics':1.3,'Transport':0.9,'Airtime':1.1,'Fashion':1.0,
    'Restaurant':0.8,'Bill Payment':1.2,'Entertainment':1.1
}
# merchant_risk_score is now a monotonic, documented transform of the SAME
# factor used to influence fraud generation -- so a model that learns
# "merchant_risk_score matters" is learning something real and internally
# consistent, not an artefact disconnected from the actual fraud mechanism.
_mf_vals = np.array(list(MERCHANT_FRAUD_FACTOR.values()))
_mf_norm = (_mf_vals - _mf_vals.min()) / (_mf_vals.max() - _mf_vals.min())
MERCHANT_RISK_SCORE = {cat: round(0.1 + 0.8*val, 6)
                        for cat, val in zip(MERCHANT_FRAUD_FACTOR.keys(), _mf_norm)}

# NOTE on composite_risk_score (RE finding #4): the original was a
# near-deterministic linear formula of features ALREADY in the model
# (merchant_risk_score, amount_vs_mean_ratio, velocity_score), causing
# double-counted / redundant signal. It is DELIBERATELY OMITTED from this
# generator rather than reproduced, since it added no independent
# information in the original.



In [ ]:
N_CUSTOMERS = 10000
DAYS = 365
START_DATE = datetime(2023, 1, 1)
print(f"Target: {N_CUSTOMERS:,} customers over {DAYS} days")

Target: 10,000 customers over 365 days


## 2. Customer and transaction generation

Strictly causal: every behavioural feature is computed from the final amount and only prior transactions -- no post-hoc replacement, no look-ahead.

In [ ]:
customers = pd.DataFrame({
    'customer_id': [f'CUST{str(i).zfill(6)}' for i in range(N_CUSTOMERS)],
    'age_group': rng.choice(AGE_GROUPS, N_CUSTOMERS),
    'location': rng.choice(LOCATIONS, N_CUSTOMERS,
                            p=[0.28,0.20,0.12,0.14,0.11,0.15]),  # roughly NIBSS-plausible weighting
})
# Each customer gets a random number of transactions across the year,
# roughly log-normal (a few very active customers, most moderate).
customers['n_tx'] = np.clip(rng.lognormal(mean=4.0, sigma=0.6, size=N_CUSTOMERS).astype(int), 5, 400)
TOTAL_TX = customers['n_tx'].sum()
print(f"Customers: {N_CUSTOMERS:,} | Planned total transactions: {TOTAL_TX:,}")

# ---------------------------------------------------------------------------
# STEP 2: Per-customer, TIME-ORDERED transaction simulation.
#
# FIX for RE findings #4, #6, #16 (the central flaw): for every transaction,
# the FINAL amount (fraud or legitimate) is decided FIRST. Every downstream
# behavioural feature (amount_vs_mean_ratio, velocity_score, amount_sum_24h,
# etc.) is then computed from that SAME final amount and from the running,
# strictly-causal transaction history. There is no post-hoc replacement step
# anywhere in this pipeline, so no internal inconsistency between "amount"
# and the features derived from it can occur.
#
# FIX for RE finding #10, #17, #18 (look-ahead leakage): amount_mean_total,
# amount_std_total, tx_count_total, channel_diversity, location_diversity,
# and online_channel_ratio are computed INCREMENTALLY, using only
# transactions up to and including the current one -- never future
# transactions. A transaction on 14 January genuinely cannot "know" about
# transactions occurring in November.
# ---------------------------------------------------------------------------

rows = []
tx_id_counter = 0

for _, cust in customers.iterrows():
    n_tx = cust['n_tx']
    cust_id = cust['customer_id']
    age_group = cust['age_group']
    location = cust['location']

    # Random, sorted timestamps across the year for this customer
    offsets = np.sort(rng.uniform(0, DAYS*24*3600, size=n_tx))
    timestamps = [START_DATE + timedelta(seconds=float(o)) for o in offsets]

    # Running causal history for this customer
    hist_amounts = []          # all amounts so far (for this customer), in time order
    hist_channels = []         # all channels so far
    hist_timestamps = []       # all timestamps so far

    for tx_i in range(n_tx):
        ts = timestamps[tx_i]
        channel = rng.choice(CHANNELS, p=[0.05,0.11,0.11,0.42,0.16,0.15])
        bank = rng.choice(BANKS)
        merchant = rng.choice(MERCHANT_CATS)
        hour = ts.hour
        month = ts.month
        dow = ts.weekday()

        # --- Decide fraud status FIRST, before amount or any feature ---
        # Deliberately NO bank/location/age effect (FIX for RE #12/13).
        fraud_prob = (BASE_FRAUD_RATE
                      * CHANNEL_FRAUD_FACTOR[channel]
                      * MONTH_FRAUD_FACTOR[month]
                      * MERCHANT_FRAUD_FACTOR[merchant])
        is_fraud = rng.random() < fraud_prob

        # --- Decide the FINAL amount (fraud or legit) BEFORE any feature ---
        if is_fraud:
            # Channel-specific fraud amount distribution (documented,
            # deliberate -- FIX for RE #9, kept but disclosed rather than
            # presented as an organic discovery). Effect size kept moderate
            # (not near-total separation from legitimate amounts).
            # Calibrated against NIBSS 2023 Annual Fraud Landscape actual reported
            # average loss per case, by channel (Businessday/NIBSS, 2023):
            # Mobile ~119,724 | Web ~101,430 | POS ~251,708 | IB ~760,082 |
            # ECOM ~58,840 | ATM ~88,089 (NGN). Lognormal mu solved so that
            # exp(mu + sigma^2/2) matches each real target mean (sigma=0.9).
            CHANNEL_FRAUD_TARGET_MEAN = {
                'Mobile':119724,'Web':101430,'POS':251708,
                'IB':760082,'ECOM':58840,'ATM':88089
            }
            sigma = 0.9
            target_mean = CHANNEL_FRAUD_TARGET_MEAN[channel]
            mu = np.log(target_mean) - (sigma**2)/2
            amount = rng.lognormal(mean=mu, sigma=sigma)
        else:
            amount = rng.lognormal(mean=10.8, sigma=0.9)
        amount = round(float(amount), 2)

        # --- NOW compute all behavioural features, from the FINAL amount,
        # using ONLY history up to and including this transaction ---
        hist_amounts.append(amount)
        hist_channels.append(channel)
        hist_timestamps.append(ts)

        amount_mean_total = float(np.mean(hist_amounts))
        amount_std_total = float(np.std(hist_amounts)) if len(hist_amounts) > 1 else 0.0
        tx_count_total = len(hist_amounts)

        # trailing 24h / 7d windows (causal: only prior + current transaction)
        window_24h = [a for a, t in zip(hist_amounts, hist_timestamps) if (ts - t).total_seconds() <= 24*3600]
        window_7d  = [a for a, t in zip(hist_amounts, hist_timestamps) if (ts - t).days <= 7]
        tx_count_24h = len(window_24h)
        amount_sum_24h = float(np.sum(window_24h))
        amount_mean_7d = float(np.mean(window_7d))
        amount_std_7d = float(np.std(window_7d)) if len(window_7d) > 1 else 0.0

        amount_vs_mean_ratio = amount / amount_mean_total if amount_mean_total > 0 else 1.0
        velocity_score = tx_count_24h * amount_sum_24h / amount_mean_total if amount_mean_total > 0 else 0.0

        channel_diversity = len(set(hist_channels))
        online_channels = {'Mobile','Web','ECOM'}
        online_channel_ratio = sum(1 for c in hist_channels if c in online_channels) / tx_count_total

        merchant_risk_score = MERCHANT_RISK_SCORE[merchant]

        fraud_technique = None
        if is_fraud:
            # FIX for RE #20: technique now correlates lightly with channel
            # (documented), rather than a pure independent draw uncorrelated
            # with channel (p=0.87 in the original).
            if channel in ('Mobile','Web'):
                fraud_technique = rng.choice(FRAUD_TECHNIQUES, p=[0.72,0.06,0.06,0.06,0.05,0.05])
            elif channel == 'ATM':
                fraud_technique = rng.choice(FRAUD_TECHNIQUES, p=[0.10,0.45,0.20,0.10,0.10,0.05])
            else:
                fraud_technique = rng.choice(FRAUD_TECHNIQUES, p=[0.55,0.12,0.12,0.09,0.07,0.05])

        rows.append({
            'transaction_id': f'TXN{str(tx_id_counter).zfill(8)}',
            'customer_id': cust_id,
            'timestamp': ts,
            'amount': amount,
            'channel': channel,
            'bank': bank,
            'merchant_category': merchant,
            'location': location,
            'age_group': age_group,
            'is_fraud': int(is_fraud),
            'fraud_technique': fraud_technique,
            'amount_log': float(np.log1p(amount)),
            'amount_vs_mean_ratio': round(amount_vs_mean_ratio, 6),
            'amount_sum_24h': round(amount_sum_24h, 2),
            'amount_mean_7d': round(amount_mean_7d, 2),
            'amount_std_7d': round(amount_std_7d, 2),
            'tx_count_24h': tx_count_24h,
            'amount_mean_total': round(amount_mean_total, 2),
            'amount_std_total': round(amount_std_total, 2),
            'tx_count_total': tx_count_total,
            'channel_diversity': channel_diversity,
            'online_channel_ratio': round(online_channel_ratio, 6),
            'velocity_score': round(velocity_score, 6),
            'merchant_risk_score': merchant_risk_score,
            'hour_sin': float(np.sin(2*np.pi*hour/24)),
            'hour_cos': float(np.cos(2*np.pi*hour/24)),
            'day_sin': float(np.sin(2*np.pi*dow/7)),
            'day_cos': float(np.cos(2*np.pi*dow/7)),
            'month_sin': float(np.sin(2*np.pi*month/12)),
            'month_cos': float(np.cos(2*np.pi*month/12)),
            'is_peak_hour': int(9 <= hour <= 18),
        })
        tx_id_counter += 1

df_new = pd.DataFrame(rows)
df_new = df_new.sort_values('timestamp').reset_index(drop=True)


Customers: 10,000 | Planned total transactions: 650,351


In [ ]:
df_new = pd.DataFrame(rows)
df_new = df_new.sort_values('timestamp').reset_index(drop=True)
print(f"Generated {len(df_new):,} transactions, {df_new['is_fraud'].sum():,} fraud "
      f"({df_new['is_fraud'].mean()*100:.3f}% fraud rate)")
df = df_new  # alias used by validation cells below

Generated 650,351 transactions, 1,943 fraud (0.299% fraud rate)


## 3. Validation - leakage fix, merchant_risk_score, bank/geography, NIBSS calibration

In [ ]:
df['reconstructed'] = df['amount_vs_mean_ratio'] * df['amount_mean_total']
df['recon_error'] = (df['reconstructed'] - df['amount']).abs()
legit_err = df.loc[df['is_fraud']==0, 'recon_error']
fraud_err = df.loc[df['is_fraud']==1, 'recon_error']
print("=== TEST 1: Amount reconstruction consistency ===")
print(f"Legit median error: NGN {legit_err.median():.2f} | Fraud median error: NGN {fraud_err.median():.2f}\n")

cat_stats = df.groupby('merchant_category').agg(fraud_rate=('is_fraud','mean'), risk_score=('merchant_risk_score','first'))
r, p = stats.pearsonr(cat_stats['risk_score'], cat_stats['fraud_rate'])
print(f"=== TEST 2: merchant_risk_score vs actual fraud rate ===\nPearson r = {r:.3f}, p = {p:.4f}\n")

chi2, p_bank, _, _ = stats.chi2_contingency(pd.crosstab(df['bank'], df['is_fraud']))
chi2_loc, p_loc, _, _ = stats.chi2_contingency(pd.crosstab(df['location'], df['is_fraud']))
print(f"=== TEST 3: Bank/location null association ===\nBank p={p_bank:.4f} | Location p={p_loc:.4f}\n")

print("=== TEST 4: Fraud amount by channel vs real NIBSS 2023 targets ===")
targets = {'Mobile':119724,'Web':101430,'POS':251708,'IB':760082,'ECOM':58840,'ATM':88089}
fraud_by_ch = df[df['is_fraud']==1].groupby('channel')['amount'].mean()
print(f"{'Channel':<10}{'Generated mean':>16}{'Real NIBSS target':>20}")
for ch, target in targets.items():
    gen = fraud_by_ch.get(ch, float('nan'))
    print(f"{ch:<10}{gen:>16,.0f}{target:>20,.0f}")

print("\n=== TEST 5: Schema check -- is_peak_hour present ===")
print('is_peak_hour' in df.columns, '| value counts:', df['is_peak_hour'].value_counts().to_dict())

print("\n=== TEST 6: composite_risk_score correctly absent ===")
print('composite_risk_score' not in df.columns)


=== TEST 1: Amount reconstruction consistency ===
Legit median error: NGN 0.02 | Fraud median error: NGN 0.02

=== TEST 2: merchant_risk_score vs actual fraud rate ===
Pearson r = 0.988, p = 0.0000

=== TEST 3: Bank/location null association ===
Bank p=0.9262 | Location p=0.8672

=== TEST 4: Fraud amount by channel vs real NIBSS 2023 targets ===
Channel     Generated mean   Real NIBSS target
Mobile             118,342             119,724
Web                 98,089             101,430
POS                231,452             251,708
IB                 728,310             760,082
ECOM                60,440              58,840
ATM                 73,054              88,089

=== TEST 5: Schema check -- is_peak_hour present ===
True | value counts: {0: 379612, 1: 270739}

=== TEST 6: composite_risk_score correctly absent ===
True


## 4. Difficulty sanity check

Confirms the leakage fix didn't overcorrect into a dataset that's trivially easy to classify.

In [ ]:
fraud = df[df['is_fraud']==1]
legit = df[df['is_fraud']==0]

print(f"Fraud: {len(fraud):,} | Legit: {len(legit):,}\n")

# TEST 1: Distributional overlap on key numeric features (KS test + visual summary stats)
print("=== TEST 1: Distribution overlap, fraud vs legit ===")
print(f"{'Feature':<25}{'Fraud median':>15}{'Legit median':>15}{'KS stat':>10}{'KS p':>12}")
key_feats = ['amount','amount_vs_mean_ratio','amount_sum_24h','velocity_score',
             'tx_count_24h','amount_mean_total','merchant_risk_score']
for feat in key_feats:
    ks, p = stats.ks_2samp(fraud[feat], legit[feat])
    print(f"{feat:<25}{fraud[feat].median():>15,.2f}{legit[feat].median():>15,.2f}{ks:>10.4f}{p:>12.2e}")

# A KS stat of 1.0 = perfect separation (bad, too easy). Near 0 = identical (fraud undetectable).
# Real fraud detection problems typically show KS in the 0.2-0.6 range for the best single feature.


Fraud: 1,943 | Legit: 648,408

=== TEST 1: Distribution overlap, fraud vs legit ===
Feature                     Fraud median   Legit median   KS stat        KS p
amount                         94,694.61      49,072.76    0.2861   7.45e-141
amount_vs_mean_ratio                1.27           0.71    0.2635   3.22e-119
amount_sum_24h                112,995.50      61,163.43    0.2580   3.45e-114
velocity_score                      1.69           0.97    0.2160    8.78e-80
tx_count_24h                        1.00           1.00    0.0110    9.72e-01
amount_mean_total              74,779.69      71,969.51    0.0920    1.06e-14
merchant_risk_score                 0.68           0.54    0.1725    7.43e-51


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
y = df['is_fraud'].values

print("=== TEST 2: Can a SINGLE feature alone solve this? (AUC-ROC, PR-AUC) ===")
print(f"{'Feature':<25}{'AUC-ROC':>10}{'PR-AUC':>10}")
for feat in ['amount','amount_vs_mean_ratio','amount_sum_24h','velocity_score','merchant_risk_score']:
    auc = roc_auc_score(y, df[feat])
    prauc = average_precision_score(y, df[feat])
    print(f"{feat:<25}{auc:>10.4f}{prauc:>10.4f}")
print("(AUC-ROC of 1.0 = perfectly solvable by this feature alone -- would be a red flag)")
print("(Real single-feature fraud signals are typically AUC 0.6-0.8, not nearly perfect)\n")

print("=== TEST 3: Simple logistic regression on amount alone ===")
X = df[['amount']].values
lr = LogisticRegression(class_weight='balanced', max_iter=1000)
lr.fit(X, y)
p = lr.predict_proba(X)[:,1]
print(f"AUC-ROC: {roc_auc_score(y, p):.4f} | PR-AUC: {average_precision_score(y, p):.4f}")
print("(A weak, single-feature model should NOT already solve fraud detection well)\n")

print("=== TEST 4: Check for any feature with suspiciously perfect correlation to is_fraud ===")
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('is_fraud')
corrs = df[numeric_cols].corrwith(df['is_fraud']).abs().sort_values(ascending=False)
print(corrs.head(10))
print("(Anything near 1.0 would mean is_fraud is essentially a copy of that feature)")


=== TEST 2: Can a SINGLE feature alone solve this? (AUC-ROC, PR-AUC) ===
Feature                     AUC-ROC    PR-AUC
amount                       0.6938    0.0177
amount_vs_mean_ratio         0.6846    0.0128
amount_sum_24h               0.6766    0.0135
velocity_score               0.6484    0.0053
merchant_risk_score          0.6137    0.0042
(AUC-ROC of 1.0 = perfectly solvable by this feature alone -- would be a red flag)
(Real single-feature fraud signals are typically AUC 0.6-0.8, not nearly perfect)

=== TEST 3: Simple logistic regression on amount alone ===
AUC-ROC: 0.6938 | PR-AUC: 0.0177
(A weak, single-feature model should NOT already solve fraud detection well)

=== TEST 4: Check for any feature with suspiciously perfect correlation to is_fraud ===
amount                  0.068736
reconstructed           0.068736
amount_sum_24h          0.056754
amount_vs_mean_ratio    0.056303
amount_mean_7d          0.046475
amount_log              0.042776
amount_std_7d           0.032

## 5. Full untuned baseline model

Confirms meaningful detection requires the actual methodology (ADASYN, hybrid architecture), not a shortcut in the data.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import xgboost as xgb
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Simple time-aware split for this sanity check (first 80% of days train, last 20% test)
cutoff = df['timestamp'].quantile(0.8)
train = df[df['timestamp'] < cutoff]
test = df[df['timestamp'] >= cutoff]

CAT = ['channel','merchant_category','bank','location','age_group']
DROP = ['transaction_id','customer_id','timestamp','is_fraud','fraud_technique']
FEAT = [c for c in df.columns if c not in DROP]
tr_enc = pd.get_dummies(train[FEAT], columns=CAT)
te_enc = pd.get_dummies(test[FEAT], columns=CAT)
tr_enc, te_enc = tr_enc.align(te_enc, join='left', axis=1, fill_value=0)

y_tr, y_te = train['is_fraud'].values, test['is_fraud'].values
sc = StandardScaler()
X_tr = sc.fit_transform(tr_enc.values.astype(float))
X_te = sc.transform(te_enc.values.astype(float))

print(f"Train: {len(y_tr):,} ({y_tr.sum()} fraud) | Test: {len(y_te):,} ({y_te.sum()} fraud)\n")

print("Training plain XGBoost (no ADASYN, no IF, no tuning -- just a sanity-check baseline)...")
model = xgb.XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.05, random_state=42, eval_metric='aucpr', n_jobs=-1, verbosity=0)
model.fit(X_tr, y_tr)
p = model.predict_proba(X_te)[:,1]

auc_roc = roc_auc_score(y_te, p)
pr_auc = average_precision_score(y_te, p)
print(f"\nFull-model AUC-ROC: {auc_roc:.4f} | PR-AUC: {pr_auc:.4f}")

for t in [0.3, 0.5, 0.7]:
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    prec = tp/(tp+fp) if tp+fp>0 else 0
    rec = tp/(tp+fn) if tp+fn>0 else 0
    print(f"theta={t}: Prec={prec:.4f} Recall={rec:.4f} TP={tp} FP={fp} FN={fn}")

print("\n(A 'too easy' dataset would show PR-AUC near 1.0 and perfect precision/recall")
print(" even with this bare-bones, untuned model. A realistic one shows real trade-offs.)")


Train: 520,280 (1555 fraud) | Test: 130,071 (388 fraud)

Training plain XGBoost (no ADASYN, no IF, no tuning -- just a sanity-check baseline)...

Full-model AUC-ROC: 0.7470 | PR-AUC: 0.0192
theta=0.3: Prec=0.2857 Recall=0.0052 TP=2 FP=5 FN=386
theta=0.5: Prec=0.3333 Recall=0.0026 TP=1 FP=2 FN=387
theta=0.7: Prec=0.0000 Recall=0.0000 TP=0 FP=1 FN=388

(A 'too easy' dataset would show PR-AUC near 1.0 and perfect precision/recall
 even with this bare-bones, untuned model. A realistic one shows real trade-offs.)


## 6. Save the dataset

Uncomment the line below to save the generated dataset to disk.

In [ ]:
# df.to_csv('NIBSS_FRAUD_DATASET_V2_CORRECTED.csv', index=False)
print("Ready to save -- uncomment the line above.")

Ready to save -- uncomment the line above.
